# Detecção de anomalias acadêmicas

Execute as células em ordem no Google Colab, em ambiente Python 3 com CPU. O notebook inclui os dados e o código do projeto.

Identificamos perfis pouco comuns para revisão a partir de G1, G2, faltas e faixa semanal de estudo. Uma anomalia estatística exige interpretação contextual.

Fonte original [Student Performance — UCI](https://archive.ics.uci.edu/dataset/320/student+performance), Paulo Cortez, DOI 10.24432/C5TG7T, licença CC BY 4.0. Usamos 649 registros e somente quatro colunas originais.

In [ ]:
from pathlib import Path
import base64, gzip, io, csv, json, shutil, subprocess

raiz = Path.cwd() / 'projeto_anomalias'
for pasta in ['dados', 'scripts', 'resultados/anomalias']:
    (raiz / pasta).mkdir(parents=True, exist_ok=True)
if shutil.which('node') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'nodejs'], check=True)
print(subprocess.check_output(['node', '--version'], text=True).strip())

## Dados e preparação

As divisões existentes foram preservadas, com 454 exemplos de treino, 97 de validação e 98 de teste. As colunas simuladas, G3 e a situação acadêmica ficam fora das entradas. A padronização usa somente média e desvio do treino. studytime representa faixas de tempo, de 1 a 4. O período das faltas exige confirmação para qualquer alerta antecipado.

O conteúdo abaixo transporta os CSVs compactados, sem download externo.

In [ ]:
dados_compactados = {
    "treino": "H4sIAAAAAAAACmVYOZLcSAz0+y0wCqiTL9A7pN0xZEjOSMb+fgM3qidCEc0hWbgykQD1DeEbwfcfnx+///n4hM8/f//978/PXx8vJMAOA/CFA3DAgPF6ABsg0AsREKHZFcGy1yY06C9s8MiFvN/4Gd9hC9jZlhxsbALlpFgjAuQTCDPurXjx8D3i9wfQ68AD9noLay2i7bDY1wbc+lRCIw5NsmpiAhtQZHD4arF9OXAADzQ2Yt79KJ94whdp8nzr6EHJj0u0pAYaKoeMU6PptQbimuyexoVPvDe17mQ5HSuLQdD5bBPfjyZiQTtixwt0okC1VMPMthKc5dbZRkY0oixSghkeyO6hgKF5HTZMGlsPUjiMlrWiV2KykugB1JIoQazylNGdoEfmus0/RTpT02ZfuMKFGN6ajwRwFEgp9TJIuzzcwIXFAMnNDjNmJBNAnkpox7WZOVQXS10oTltglFCOtwoGQ+Vfzbtc9WjLpqYv8lxXU3mLNxgnCJWQol0+0idWDj/SC4+tlSkISEbAFAqn7tLYjMROe64UV3bFsyTHKCA6AQkDguV8HkWBTJRWFARHIOUs0q716KhKxAhstbHaV7o1d+tGtIRPRExBxea93vXgiIcecDgwDK1l4/HFQEsoK9CiQKo5jmuPJjNMsDrOFsRoVZFGkuo90QErWL5ghbURzJHU5KR3StN27yG02fjN6qkBbJgWqCSClZU9EEL1VwqvpFkWYJJ3Qm/WVEvl0P156lj7oRu+ybVlp3fARMFq4ytLx4kDVA6kzitIdDWVd82wuXAyITLGVVzQROptJjaRn7tBxWwnSBkYARzWaRJT7pbYrk9nDOqUtsVExK986ToM8F2p8XbyXKP6WhCGTzGxz/5e2wDCOgtj+DolbdvAGnryMMdJV3ioaMcRXafQVwNgyqMnwqfLn8ac0kox/n0R0MlZ94TU5aUdVTWYwcfhALtEdQh8sV1Ry/urVM9HJLqSEadDtWojDJJLnmkpVYSEJLvowrCx7t096uZmEaeSDiNVS2AIY+3BGKmOy7wpkCTMvcp8joKRCcY93qmOUiorx41vjdy1YyhtrkUDQ8ln3QkbLN8WWngITU9Ge0lGFK4XuUVXPowZRYrocfppT60IxblMxXRlBdUKYEz843uDL2J1NLaqnhQqZUt0iwr0WLF73ebSreUmLqZtTlsFQTL3PbSLeEgPzDJNbXJSsZJbb8rucb7VZRNrfUzsYqqcL5GaoKuVYZWaV3eVAAje9iWMnu4M1TWjbBaUTqKQUZP75GOOivg0yKexReS3R+5hKzfwJ0DpsV+tO26Vz/V20LunVfnA95ypKmflWM7gmn1dOcqeNmyJ9ZFWt/Iepn3arAAlV/sTpnOzKiLkUY04OmJMrKj6XUX/LqxLkn8pPlb6rXXb/inB1pq9Mf2jgGxSIKPP+47eCFIf6EKV2BAeSWzDlqF9TBeP6w6b4PD5+8hfoHQ2xKTuxvtW4i0bhL2Zm5VPOdO+Y9uxWuUgtpWjXZ/L24qg43Hbr4yGyPBAbtz8zBBf4kQ1n4ZZOeaNzNqqH+Vc61n/G2Db51bcYMdD/NoH0YFVV2n12iJ0NUHshVtZbU6rsP+9r185yCcnZFT6uyIo1BeWQDHNpUeb0+N/OoY6ey0RAAA=",
    "validacao": "H4sIAAAAAAAAClVTO3YEIQzr5ywq8AcGTrDnyGeLFEmzmyK3z/MHlqnwYNnIsuZGuDHe3h/3n4/7A4/n7+ff8+v7fgxQgYIOUpCALaqghgI5qFi2gA8Sy3rEIAJphuLAHgHlFYOPE8NP6yFwuE44FYyAC4gjSda2JIxXaUEzHAVNNr4UBSeoB0yNTssZ1F9wGJFl2z4N+awDAzWf162Ho5KI+F0NmmUO4wW6NYsHNJgHbPagljMQJKXxgcS1CR47y7rNpVnKG/PJKZXWyb1juGK5Go/qxnOgL/VfspbctOYus+K1VU4BGJp3GlJkcV/euGhXNnYenaAzmtCUbARsCsUXL22mSyO+mIQT0k6ydNfF+HJXg7EboC2P8dxUMU4SGkeSlypZEOKRL6y71NVP+27u+5EDDWfX0dP8pze104xZs87ymmf07RD/LpNE/BgVoeM//trHLLgDAAA=",
    "teste": "H4sIAAAAAAAAClVTOXLDQAzr9RYUC+6pF/gdOVykSBo7RX6f4SFamvHYO+ABAqRvxE3w9v64/3zcH3g8fz//nl/f940ECwpk27GjgBsFFAhkY4V+CurGBWq4bexgt/AOMvMauqYVxawLtVb7ktixAoqCoT0iyI6lpUY7FOserVHRorZAcjzDuo0nERZvWM+v5okTXN4wXow8VbJNd+DVxUmq1xYnPtwoV0xyQGPT7zShhk5LbKm4gTPViTsj56ErqD7nYo4KOXnvHBK+VffN3BrZjxJgSTA2Yi+JAQeOvJqLqNqJl5PopvlMbLW0m2gq/mzhoa7lSg4Xgm6mJoH3ZJ7ApU1xh2tO/TL4YohtYuWprEvens4UT5NY4jCVFmzBKiloxfB0zBjYwiwfb2Ha3s0qjuOhyMRAtfNaCOd293Bht5tZB6AplPgP2kEPJYfYb7HUFUL8Jmb8/gNnIj8W2AMAAA=="
}

for parte, conteudo in dados_compactados.items():
    dados = gzip.decompress(base64.b64decode(conteudo))
    arquivo = raiz / 'dados' / f'{parte}.csv'
    arquivo.write_bytes(dados)
    registros = list(csv.DictReader(io.StringIO(dados.decode())))
    print(parte, len(registros), 'registros')

## Rede, baseline e treinamento

O autoencoder 4 → 2 → 4 comprime e reconstrói as entradas. A camada oculta usa tanh e a saída é linear. A perda é o erro quadrático médio. Usamos descida do gradiente estocástica, lote 1, taxa 0,005 e semente 42. A menor perda da validação seleciona os pesos, com parada após 60 épocas sem melhora.

A baseline mede distância ao perfil médio por média dos quadrados das entradas padronizadas. Cada método sinaliza pontuação acima de seu percentil 95 na validação. O teste é lido após fixar os pesos e os limites.

In [ ]:
%%writefile projeto_anomalias/scripts/anomalias.mjs
import fs from 'node:fs';
import path from 'node:path';
import { fileURLToPath } from 'node:url';

export const campos = ['G1', 'G2', 'absences', 'studytime'];
const raiz = path.resolve(path.dirname(fileURLToPath(import.meta.url)), '..');
const pasta = path.join(raiz, 'resultados/anomalias');
const ler = (arquivo) => fs.readFileSync(path.join(raiz, arquivo), 'utf8').replace(/^\uFEFF/, '');
const gravar = (nome, valor) => fs.writeFileSync(path.join(pasta, nome), JSON.stringify(valor, null, 2) + '\n');

export function carregar(parte) {
    const linhas = ler(`dados/${parte}.csv`).trim().split(/\r?\n/);
    const separar = (linha) => linha.split(',').map((valor) => valor.replace(/^"|"$/g, ''));
    const nomes = separar(linhas.shift());
    const indices = campos.map((campo) => nomes.indexOf(campo));
    if (indices.includes(-1)) throw new Error('Coluna original ausente.');
    return linhas.map((linha, i) => {
        const valores = separar(linha);
        const x = indices.map((indice) => Number(valores[indice]));
        if (!x.every(Number.isFinite)) throw new Error('Entrada inválida.');
        return { linha: i + 2, x };
    });
}

export function ajustarEscala(dados) {
    return campos.map((_, j) => {
        const media = dados.reduce((soma, item) => soma + item.x[j], 0) / dados.length;
        const variancia = dados.reduce((soma, item) => soma + (item.x[j] - media) ** 2, 0) / dados.length;
        return { media, desvio: Math.sqrt(variancia) || 1 };
    });
}

export function padronizar(x, escala) {
    return x.map((valor, j) => (valor - escala[j].media) / escala[j].desvio);
}

export function percentil95(valores) {
    const ordenados = [...valores].sort((a, b) => a - b);
    const posicao = 0.95 * (ordenados.length - 1);
    const i = Math.floor(posicao);
    return ordenados[i] + (posicao - i) * (ordenados[Math.ceil(posicao)] - ordenados[i]);
}

export class Autoencoder {
    constructor() {
        let estado = 42;
        const sortear = () => {
            estado = (Math.imul(1664525, estado) + 1013904223) >>> 0;
            return estado / 4294967296;
        };
        this.codificador = Array.from({ length: 2 }, () => campos.map(() => (sortear() - 0.5) * 0.5));
        this.decodificador = campos.map(() => [0, 0].map(() => (sortear() - 0.5) * 0.5));
        this.viesOculto = [0, 0];
        this.viesSaida = campos.map(() => 0);
    }

    reconstruir(x) {
        const oculto = this.codificador.map((pesos, i) => Math.tanh(
            pesos.reduce((soma, peso, j) => soma + peso * x[j], this.viesOculto[i])));
        const saida = this.decodificador.map((pesos, i) => pesos.reduce(
            (soma, peso, j) => soma + peso * oculto[j], this.viesSaida[i]));
        return { oculto, saida };
    }

    pontuar(x) {
        return this.reconstruir(x).saida.reduce((soma, valor, j) => soma + (valor - x[j]) ** 2, 0) / x.length;
    }

    atualizar(x, taxa) {
        const { oculto, saida } = this.reconstruir(x);
        const deltaSaida = saida.map((valor, j) => 2 * (valor - x[j]) / x.length);
        const deltaOculto = oculto.map((valor, j) => (1 - valor ** 2) * this.decodificador.reduce(
            (soma, pesos, i) => soma + pesos[j] * deltaSaida[i], 0));
        this.decodificador.forEach((pesos, i) => {
            pesos.forEach((_, j) => { pesos[j] -= taxa * deltaSaida[i] * oculto[j]; });
            this.viesSaida[i] -= taxa * deltaSaida[i];
        });
        this.codificador.forEach((pesos, i) => {
            pesos.forEach((_, j) => { pesos[j] -= taxa * deltaOculto[i] * x[j]; });
            this.viesOculto[i] -= taxa * deltaOculto[i];
        });
    }
}

export const cenarios = [
    { nome: 'queda_notas', elegivel: ([g1, g2]) => g1 >= 12 && g2 >= 10,
        alterar: ([g1, , faltas, estudo]) => [g1, 0, faltas, estudo] },
    { nome: 'faltas_notas_altas', elegivel: ([g1, g2, faltas]) => g1 >= 15 && g2 >= 15 && faltas <= 10,
        alterar: ([g1, g2, , estudo]) => [g1, g2, 30, estudo] },
    { nome: 'estudo_notas_baixas', elegivel: ([g1, g2, , estudo]) => g1 <= 9 && g2 <= 9 && estudo <= 2,
        alterar: ([g1, g2, faltas]) => [g1, g2, faltas, 4] }
];

export function executar() {
    fs.mkdirSync(pasta, { recursive: true });
    const treino = carregar('treino');
    const validacao = carregar('validacao');
    const escala = ajustarEscala(treino);
    const xTreino = treino.map((item) => padronizar(item.x, escala));
    const xValidacao = validacao.map((item) => padronizar(item.x, escala));
    const modelo = new Autoencoder();
    const configuracao = { semente: 42, taxa: 0.005, lote: 1, max_epocas: 1000, paciencia: 60 };
    const perda = (dados) => dados.reduce((soma, x) => soma + modelo.pontuar(x), 0) / dados.length;
    let melhorPerda = Infinity;
    let melhorEpoca = 0;
    let pesos;
    let estado = 42;
    const historico = [];
    const inicio = performance.now();
    for (let epoca = 1; epoca <= configuracao.max_epocas; epoca++) {
        const ordem = [...xTreino];
        for (let i = ordem.length - 1; i > 0; i--) {
            estado = (Math.imul(1664525, estado) + 1013904223) >>> 0;
            const j = Math.floor(estado / 4294967296 * (i + 1));
            [ordem[i], ordem[j]] = [ordem[j], ordem[i]];
        }
        for (const x of ordem) modelo.atualizar(x, configuracao.taxa);
        const registro = { epoca, treino: perda(xTreino), validacao: perda(xValidacao) };
        if (!Number.isFinite(registro.validacao)) throw new Error('Treinamento divergiu.');
        historico.push(registro);
        if (registro.validacao < melhorPerda - 1e-8) {
            melhorPerda = registro.validacao;
            melhorEpoca = epoca;
            pesos = structuredClone(modelo);
        }
        if (epoca - melhorEpoca >= configuracao.paciencia) break;
    }
    Object.assign(modelo, pesos);
    const segundos = (performance.now() - inicio) / 1000;
    const metodos = {
        autoencoder: (x) => modelo.pontuar(x),
        baseline: (x) => x.reduce((soma, valor) => soma + valor ** 2, 0) / x.length
    };
    const limites = Object.fromEntries(Object.entries(metodos).map(([nome, pontuar]) =>
        [nome, percentil95(xValidacao.map(pontuar))]));
    // O teste é lido após a seleção dos pesos e dos limites.
    const teste = carregar('teste');
    const originais = teste.map((item) => {
        const x = padronizar(item.x, escala);
        return { linha_csv: item.linha, valores: item.x,
            ...Object.fromEntries(Object.entries(metodos).map(([nome, pontuar]) => {
                const pontuacao = pontuar(x);
                return [nome, { pontuacao, sinalizado: pontuacao > limites[nome] }];
            })) };
    });
    const resultados = cenarios.map((cenario) => {
        const pares = teste.filter((item) => cenario.elegivel(item.x)).map((item) => {
            const alterado = cenario.alterar(item.x);
            return { linha_csv: item.linha, original: item.x, alterado,
                ...Object.fromEntries(Object.entries(metodos).map(([nome, pontuar]) => {
                    const antes = pontuar(padronizar(item.x, escala));
                    const depois = pontuar(padronizar(alterado, escala));
                    return [nome, { antes, depois, variacao: depois - antes, detectado: depois > limites[nome] }];
                })) };
        });
        return { cenario: cenario.nome, elegiveis: pares.length,
            resumo: Object.fromEntries(Object.keys(metodos).map((nome) => [nome, {
                taxa_deteccao: pares.length ? pares.filter((p) => p[nome].detectado).length / pares.length : null,
                variacao_media: pares.length ? pares.reduce((soma, p) => soma + p[nome].variacao, 0) / pares.length : null
            }])), pares };
    });
    const resumo = { campos, arquitetura: [4, 2, 4], configuracao, melhor_epoca: melhorEpoca,
        epocas_executadas: historico.length, segundos_treinamento: segundos, limites,
        registros: { treino: treino.length, validacao: validacao.length, teste: teste.length },
        teste_original: Object.fromEntries(Object.keys(metodos).map((nome) => [nome, {
            sinalizados: originais.filter((item) => item[nome].sinalizado).length,
            proporcao: originais.filter((item) => item[nome].sinalizado).length / teste.length
        }])), cenarios: resultados.map(({ pares, ...item }) => item) };
    gravar('modelo.json', { campos, escala, limites, pesos: modelo });
    gravar('historico.json', historico);
    gravar('teste_original.json', originais);
    gravar('avaliacao_controlada.json', resultados);
    gravar('resumo.json', resumo);
    console.log(JSON.stringify(resumo, null, 2));
}

if (process.argv[1] && path.resolve(process.argv[1]) === fileURLToPath(import.meta.url)) executar();


In [ ]:
execucao = subprocess.run(['node', 'scripts/anomalias.mjs'], cwd=raiz, capture_output=True, text=True, check=True)
print(execucao.stdout)

## Resultados e comparação

Os registros originais não possuem rótulos confirmados de anomalia. Por isso, a proporção sinalizada não mede falsos positivos. Nas cópias do teste, aplicamos os três cenários definidos no protocolo. Eles medem sensibilidade a alterações didáticas.

In [ ]:
resumo = json.loads((raiz / 'resultados/anomalias/resumo.json').read_text())
print('Melhor época', resumo['melhor_epoca'])
print('Épocas executadas', resumo['epocas_executadas'])
print('Tempo de treinamento (s)', round(resumo['segundos_treinamento'], 3))
for metodo, resultado in resumo['teste_original'].items():
    print(metodo, resultado['sinalizados'], 'de 98 registros originais sinalizados')
print('\nCenário | Exemplos | Autoencoder | Baseline')
for cenario in resumo['cenarios']:
    taxas = [cenario['resumo'][m]['taxa_deteccao'] for m in ['autoencoder', 'baseline']]
    textos = ['indisponível' if t is None else f'{t:.1%}' for t in taxas]
    print(cenario['cenario'], cenario['elegiveis'], *textos, sep=' | ')

In [ ]:
import matplotlib.pyplot as plt

historico = json.loads((raiz / 'resultados/anomalias/historico.json').read_text())
fig, ax = plt.subplots(figsize=(8, 4))
for parte in ['treino', 'validacao']:
    ax.plot([h['epoca'] for h in historico], [h[parte] for h in historico], label=parte)
ax.axvline(resumo['melhor_epoca'], color='gray', linestyle='--', label='época selecionada')
ax.set(xlabel='Época', ylabel='Erro quadrático médio', title='Reconstrução das entradas')
ax.legend()
plt.show()

## Detecções e alterações não detectadas

As linhas indicadas pertencem aos CSVs de cada divisão, contando o cabeçalho como linha 1. Cada cópia mantém a referência ao registro de origem. Uma alteração detectada não comprova dificuldade acadêmica real.

In [ ]:
avaliacao = json.loads((raiz / 'resultados/anomalias/avaliacao_controlada.json').read_text())
for cenario in avaliacao:
    print('\n', cenario['cenario'])
    for detectado in [True, False]:
        exemplo = next((p for p in cenario['pares'] if p['autoencoder']['detectado'] == detectado), None)
        if exemplo is None:
            print('Sem exemplo', 'detectado' if detectado else 'não detectado')
            continue
        print('Detectado' if detectado else 'Não detectado', '— linha', exemplo['linha_csv'])
        print('Original', dict(zip(resumo['campos'], exemplo['original'])))
        print('Alterado', dict(zip(resumo['campos'], exemplo['alterado'])))

## Verificação dos cálculos

Conferimos gradientes por diferenças finitas, percentil, padronização, faixas das alterações, preservação dos originais e carregamento dos pesos.

In [ ]:
%%writefile projeto_anomalias/scripts/verificar_anomalias.mjs
import assert from 'node:assert/strict';
import fs from 'node:fs';
import { Autoencoder, carregar, ajustarEscala, padronizar, percentil95, cenarios } from './anomalias.mjs';

const modelo = new Autoencoder();
const x = [0.2, -0.7, 1.1, 0.4];
const original = structuredClone(modelo);
const taxa = 0.0001;
const epsilon = 0.000001;
modelo.atualizar(x, taxa);
for (const campo of ['codificador', 'decodificador', 'viesOculto', 'viesSaida']) {
    const matriz = Array.isArray(original[campo][0]);
    const linhas = matriz ? original[campo] : [original[campo]];
    for (let i = 0; i < linhas.length; i++) {
        for (let j = 0; j < linhas[i].length; j++) {
            const copiar = () => Object.assign(new Autoencoder(), structuredClone(original));
            const mais = copiar();
            const menos = copiar();
            if (matriz) {
                mais[campo][i][j] += epsilon;
                menos[campo][i][j] -= epsilon;
            } else {
                mais[campo][j] += epsilon;
                menos[campo][j] -= epsilon;
            }
            const numerico = (mais.pontuar(x) - menos.pontuar(x)) / (2 * epsilon);
            const atualizado = matriz ? modelo[campo][i][j] : modelo[campo][j];
            const analitico = (linhas[i][j] - atualizado) / taxa;
            assert.ok(Math.abs(numerico - analitico) < 1e-6, `Gradiente incorreto em ${campo}`);
        }
    }
}
assert.equal(percentil95([0, 10, 20, 30, 40]), 38);
const treino = carregar('treino');
const escala = ajustarEscala(treino);
for (let j = 0; j < 4; j++) {
    const valores = treino.map((item) => padronizar(item.x, escala)[j]);
    assert.ok(Math.abs(valores.reduce((a, b) => a + b, 0) / valores.length) < 1e-12);
    assert.ok(Math.abs(valores.reduce((a, b) => a + b * b, 0) / valores.length - 1) < 1e-12);
}
for (const item of carregar('teste')) {
    for (const cenario of cenarios) {
        if (!cenario.elegivel(item.x)) continue;
        const antes = [...item.x];
        const alterado = cenario.alterar(item.x);
        assert.deepEqual(item.x, antes);
        assert.ok(alterado[0] >= 0 && alterado[0] <= 20 && alterado[1] >= 0 && alterado[1] <= 20);
        assert.ok(alterado[2] >= 0 && alterado[2] <= 93 && alterado[3] >= 1 && alterado[3] <= 4);
    }
}
const salvo = JSON.parse(fs.readFileSync(new URL('../resultados/anomalias/modelo.json', import.meta.url), 'utf8'));
assert.deepEqual(salvo.escala, escala);
const recarregado = Object.assign(new Autoencoder(), salvo.pesos);
assert.ok(Number.isFinite(recarregado.pontuar(x)));
console.log('Verificações aprovadas: gradientes, percentil, normalização, alterações e modelo salvo.');


In [ ]:
verificacao = subprocess.run(['node', 'scripts/verificar_anomalias.mjs'], cwd=raiz, capture_output=True, text=True, check=True)
print(verificacao.stdout)

## Limitações e conclusão

O autoencoder empatou com a baseline nos cenários de queda de notas e faltas elevadas, e detectou menos alterações no cenário de estudo elevado com notas baixas. O experimento não demonstrou vantagem da rede.

A base pequena, a ausência de rótulos de anomalia, as faixas ordinais de estudo e a falta de datas confirmadas limitam a aplicação. Os casos alterados podem representar combinações plausíveis. A validação seleciona tanto os pesos quanto o limite, limitando a independência da calibração. Os resultados exigem revisão contextual, proteção da privacidade e avaliação independente antes de uso institucional.

In [ ]:
arquivo = shutil.make_archive(str(raiz / 'resultados_anomalias'), 'zip', raiz / 'resultados/anomalias')
print('Resultados salvos em', arquivo)

## Quais registros foram sinalizados?

A tabela identifica cada caso pela linha do CSV de teste e mostra a variável com maior contribuição para o erro. Os detalhes numéricos ficam em anomalias_explicadas.json.

In [ ]:
import json
import math
from pathlib import Path

raiz = Path.cwd() / 'projeto_anomalias'
pasta = raiz / 'resultados' / 'anomalias'
modelo = json.loads((pasta / 'modelo.json').read_text(encoding='utf-8-sig'))
registros = json.loads((pasta / 'teste_original.json').read_text(encoding='utf-8-sig'))
nomes = ['Nota G1', 'Nota G2', 'Faltas', 'Faixa de estudo']
faixas = {1: 'menos de 2 h', 2: '2 a 5 h', 3: '5 a 10 h', 4: 'mais de 10 h'}
pesos = modelo['pesos']
limite = modelo['limites']['autoencoder']
casos = []
for registro in registros:
    if not registro['autoencoder']['sinalizado']:
        continue
    valores = registro['valores']
    x = [(v - e['media']) / e['desvio'] for v, e in zip(valores, modelo['escala'])]
    oculto = [math.tanh(sum(p * v for p, v in zip(linha, x)) + vies)
              for linha, vies in zip(pesos['codificador'], pesos['viesOculto'])]
    saida = [sum(p * v for p, v in zip(linha, oculto)) + vies
             for linha, vies in zip(pesos['decodificador'], pesos['viesSaida'])]
    erros = [(real - reconstruido) ** 2 for real, reconstruido in zip(x, saida)]
    total = sum(erros)
    ordem = sorted(range(4), key=lambda j: erros[j], reverse=True)
    detalhes = []
    for j in ordem:
        reconstruido = saida[j] * modelo['escala'][j]['desvio'] + modelo['escala'][j]['media']
        detalhes.append({'variavel': nomes[j], 'valor': valores[j],
                         'reconstrucao': reconstruido, 'erro_padronizado_quadrado': erros[j],
                         'contribuicao_percentual': 100 * erros[j] / total if total else 0})
    principal = detalhes[0]
    direcao = 'acima' if principal['valor'] > principal['reconstrucao'] else 'abaixo'
    explicacao = (f"{principal['variavel']} {direcao} da reconstrução, com "
                  f"{principal['contribuicao_percentual']:.1f}% do erro total.")
    casos.append({'linha_teste_csv': registro['linha_csv'], 'G1': valores[0], 'G2': valores[1],
                  'faltas': valores[2], 'estudo_semanal': faixas[int(valores[3])],
                  'pontuacao': registro['autoencoder']['pontuacao'], 'limite': limite,
                  'explicacao': explicacao, 'detalhes': detalhes})
casos.sort(key=lambda caso: caso['pontuacao'], reverse=True)
(pasta / 'anomalias_explicadas.json').write_text(json.dumps(casos, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
linhas = ['Registros sinalizados para revisão',
          f'Limite de pontuação {limite:.4f}. Identificação pela linha de dados/teste.csv.',
          'A reconstrução descreve o modelo. Valores reconstruídos podem ser fracionários ou ultrapassar faixas originais.',
          'A contribuição informa qual variável aumentou o erro. A causa real exige revisão contextual.', '']
for caso in casos:
    linhas.append(f"Linha {caso['linha_teste_csv']} | G1 {caso['G1']} | G2 {caso['G2']} | "
                  f"Faltas {caso['faltas']} | Estudo {caso['estudo_semanal']} | "
                  f"Pontuação {caso['pontuacao']:.4f} | {caso['explicacao']}")
relatorio = '\n'.join(linhas) + '\n'
(pasta / 'anomalias_explicadas.txt').write_text(relatorio, encoding='utf-8')
print(relatorio)

from IPython.display import display, HTML
import html
colunas = ['linha_teste_csv', 'G1', 'G2', 'faltas', 'estudo_semanal', 'pontuacao', 'explicacao']
cabecalhos = ['Linha no teste', 'G1', 'G2', 'Faltas', 'Estudo semanal', 'Pontuação', 'Principal contribuição']
html_tabela = '<table><tr>' + ''.join('<th>' + c + '</th>' for c in cabecalhos) + '</tr>'
for caso in casos:
    html_tabela += '<tr>'
    for coluna in colunas:
        valor = f"{caso[coluna]:.4f}" if coluna == 'pontuacao' else str(caso[coluna])
        html_tabela += '<td>' + html.escape(valor) + '</td>'
    html_tabela += '</tr>'
display(HTML(html_tabela + '</table>'))


## Demonstração com um novo aluno

Preencha as quatro entradas na célula abaixo e execute. A análise reutiliza os pesos, a escala e o limite do treinamento. A faixa de estudo usa os códigos de 1 a 4 na tabela de reconstrução.

In [ ]:
#@title Analisar um novo aluno
nota_g1 = 15 #@param {type:"number"}
nota_g2 = 2 #@param {type:"number"}
faltas = 4 #@param {type:"integer"}
faixa_estudo = "2 a 5 horas" #@param ["Menos de 2 horas", "2 a 5 horas", "5 a 10 horas", "Mais de 10 horas"]

import json
import math
import subprocess
from pathlib import Path
from IPython.display import display, HTML

raiz = Path.cwd() / 'projeto_anomalias'
faixas = ["Menos de 2 horas", "2 a 5 horas", "5 a 10 horas", "Mais de 10 horas"]
if not all(isinstance(n, (int, float)) and math.isfinite(n) and 0 <= n <= 20
           for n in [nota_g1, nota_g2]):
    raise ValueError('As notas devem estar entre 0 e 20.')
if not isinstance(faltas, (int, float)) or not math.isfinite(faltas) or not 0 <= faltas <= 93 or faltas != int(faltas):
    raise ValueError('As faltas devem ser um número inteiro entre 0 e 93.')
if faixa_estudo not in faixas:
    raise ValueError('Selecione uma das quatro faixas de estudo.')
if not (raiz / 'resultados/anomalias/modelo.json').exists():
    raise FileNotFoundError('Execute primeiro as células de preparação e treinamento.')

valores = [nota_g1, nota_g2, int(faltas), faixas.index(faixa_estudo) + 1]
codigo = r"""
import fs from 'node:fs';
import { Autoencoder, padronizar } from './scripts/anomalias.mjs';
const salvo = JSON.parse(fs.readFileSync('resultados/anomalias/modelo.json', 'utf8'));
const valores = JSON.parse(fs.readFileSync(0, 'utf8'));
const rede = Object.assign(new Autoencoder(), salvo.pesos);
const x = padronizar(valores, salvo.escala);
const saida = rede.reconstruir(x).saida;
const erros = saida.map((v, j) => (v - x[j]) ** 2);
const total = erros.reduce((soma, v) => soma + v, 0);
const pontuacao = rede.pontuar(x);
const detalhes = salvo.campos.map((campo, j) => ({
    campo, informado: valores[j],
    reconstrucao: saida[j] * salvo.escala[j].desvio + salvo.escala[j].media,
    contribuicao: total > 0 ? 100 * erros[j] / total : 0
})).sort((a, b) => b.contribuicao - a.contribuicao);
console.log(JSON.stringify({pontuacao, limite: salvo.limites.autoencoder,
    sinalizado: pontuacao > salvo.limites.autoencoder, detalhes}));
"""
execucao = subprocess.run(['node', '--input-type=module', '-e', codigo], cwd=raiz,
                         input=json.dumps(valores), capture_output=True, text=True, check=True)
resultado = json.loads(execucao.stdout)
nomes = {'G1': 'Nota G1', 'G2': 'Nota G2', 'absences': 'Faltas', 'studytime': 'Faixa de estudo'}
principal = resultado['detalhes'][0]
status = 'Possível anomalia — revisar o perfil' if resultado['sinalizado'] else 'Perfil abaixo do limite de sinalização'
display(HTML(f"<h3>{status}</h3><p>Pontuação <b>{resultado['pontuacao']:.4f}</b> · "
             f"Limite <b>{resultado['limite']:.4f}</b></p>"))
if resultado['sinalizado']:
    direcao = 'acima' if principal['informado'] > principal['reconstrucao'] else 'abaixo'
    print(f"Maior contribuição: {nomes[principal['campo']]} {direcao} da reconstrução, "
          f"com {principal['contribuicao']:.1f}% do erro total.")
print(f'Entradas: G1 {nota_g1}, G2 {nota_g2}, faltas {int(faltas)}, estudo {faixa_estudo}.')
tabela = '<table><tr><th>Variável</th><th>Informado</th><th>Reconstrução</th><th>Parcela do erro</th></tr>'
for item in resultado['detalhes']:
    tabela += (f"<tr><td>{nomes[item['campo']]}</td><td>{item['informado']}</td>"
               f"<td>{item['reconstrucao']:.2f}</td><td>{item['contribuicao']:.1f}%</td></tr>")
display(HTML(tabela + '</table>'))
print('A reconstrução é uma aproximação do modelo e pode conter valores fracionários ou fora das faixas originais.')
print('O sinal exige análise contextual. Um perfil abaixo do limite também pode exigir acompanhamento.')
